Clustering behavior for input activations for randomly sampled points v/s points which are high contribs.  
I will do this for only `layers.2.o12` layer.  

I reached this because I wanted to cluster kernel patches to do automatic group detection without me constantly watching them, but did not get good results on random points taken from the input.  

Ofcourse, first we analyse, i might be fully wrong.  


There is definite problem with my own contrib finding strategy. I see really good clustering results on (2,3), (2,4) and (2,5) for the whole channel.  
It seems ill need to for now, take the max contribs for each poi, and then do something about them.   
It would been useful to have some sort of easy way to pick them.  

Problems with auto clustering:
- I dont know which contribs to pick. It seems to follow a pattern per POI, so it would be interesting to get a cut-off per POI
- I don't know the statistics of what a good cluster is
- Random picking is full of 0s, so clustering is giving good results for random pickings.  



In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
import os
import sys
import django

# Setup Django environment
# Adjust the path to point to the directory containing manage.py
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../hiccup_ide")))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "hiccup_ide.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.utils import show_single_channel_red_green_black as S, to_show_list as tsl
from neural_data.models import *

In [ ]:
# we have only 151 examples for now, ill add more later
# these are all mostly 4s though but its okay
Activation.objects.filter(input__alias="4-Sl1", coordinate__startswith="layers.1").values_list("coordinate", flat=True)

In [ ]:
from sklearn.decomposition import PCA

def do_pca(dims, lin_patches):
    # 14 is fine, found after doing for 20
    # weird that the value i need to set is higher actually
    pca = PCA(dims)
    X_reduced = pca.fit_transform(lin_patches)


    exp_var_pca = pca.explained_variance_ratio_
    cum_sum_eigenvalues = np.cumsum(exp_var_pca)

    # 5. Create the Scree Plot
    plt.figure(figsize=(10, 6))

    # Individual variance bars
    plt.bar(range(1, len(exp_var_pca) + 1), exp_var_pca, alpha=0.5, align='center',
            label='Individual explained variance')

    # Cumulative variance step plot
    plt.step(range(1, len(cum_sum_eigenvalues) + 1), cum_sum_eigenvalues, where='mid',
            label='Cumulative explained variance', color='red')

    plt.ylabel('Explained variance ratio')
    plt.xlabel('Principal component index')
    plt.title('Scree Plot: Explained Variance by Components')
    plt.xticks(range(1, len(exp_var_pca) + 1))
    plt.legend(loc='best')
    plt.tight_layout()
    plt.show()
    return X_reduced

In [ ]:
Activation.objects.first().input

In [ ]:
# first we need input activations for the kernel (we do layer 2, so all input activations to that layer for each input)
# we take a random cube first out of all activations

from collections import defaultdict

ip2act = {}
for ip in Input.objects.all():
    acts = list(Activation.objects.filter(coordinate__startswith="layers.1.", input=ip).order_by("coordinate"))
    ip2act[ip.alias] = np.stack([np.array(a.data) for a in acts])

S([n for n in ip2act["4-Sl1"]], 10, ncols=4)

In [ ]:
weights = list(Weight.objects.filter(coordinate__startswith="layers.2.out_12", data_type="weights").order_by("coordinate"))
_weights = [np.array(w.data) for w in weights]
kernel = np.stack(_weights)

S([kernel[0], _weights[0]])

In [ ]:
def get_receptive(y, x, ksize=3, stride=2, padding=1):
    ys = y*stride - padding
    xs = x*stride - padding
    return (ys, xs), (ys+ksize, xs+ksize)

In [ ]:
# we first sample the activations we want, we are looking at the output of layers.2.ch12

d = SaliencyMap.objects.filter(coordinate="layers.2.out_12").first().data
out_h, out_w = len(d), len(d[0])
out_h, out_w

In [ ]:
from tqdm import tqdm

def get_pos(sm):
    res = []
    for r, row in enumerate(sm.data):
        for c, col in enumerate(row):
            if sm.data[r][c] > 0:
                res.append((col, (r, c), sm))
    return res

pos_sms = []
sms = list(SaliencyMap.objects.filter(coordinate="layers.2.out_12"))
# print(len(sms))
for sm in tqdm(sms):
    pos_sms.extend(get_pos(sm))

In [ ]:
plt.hist([a[0] for a in pos_sms])
plt.show()

In [ ]:
gt_002 = [a for a in pos_sms if a[0] > 0.02]
len(gt_002)

In [ ]:
import random
POINT_COUNT = len(gt_002)
def _rand_point(H, W):
    return random.randint(1, H-1), random.randint(1, W-1)

# Random points clustering

In [ ]:
patches = []
for alias, act in ip2act.items():
    y,x = _rand_point(out_h, out_w)
    (y0,x0), (y1,x1) = get_receptive(y,x)
    patches.append(act[:, y0:y1, x0:x1])

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
lin_patches = [p.reshape(-1) for p in patches]
pws = lin_patches * kernel.reshape(-1)
scaled_pws = scaler.fit_transform(pws)

In [ ]:
from sklearn.decomposition import PCA

# 7 is fine, found after doing for 20
pca = PCA(20)
X_reduced = pca.fit_transform(scaled_pws)


exp_var_pca = pca.explained_variance_ratio_
cum_sum_eigenvalues = np.cumsum(exp_var_pca)

# 5. Create the Scree Plot
plt.figure(figsize=(10, 6))

# Individual variance bars
plt.bar(range(1, len(exp_var_pca) + 1), exp_var_pca, alpha=0.5, align='center',
        label='Individual explained variance')

# Cumulative variance step plot
plt.step(range(1, len(cum_sum_eigenvalues) + 1), cum_sum_eigenvalues, where='mid',
         label='Cumulative explained variance', color='red')

plt.ylabel('Explained variance ratio')
plt.xlabel('Principal component index')
plt.title('Scree Plot: Explained Variance by Components')
plt.xticks(range(1, len(exp_var_pca) + 1))
plt.legend(loc='best')
plt.tight_layout()
plt.show()

In [ ]:
import scipy.cluster.hierarchy as sch
import matplotlib.pyplot as plt

# linkage performs the actual clustering
linkage_matrix = sch.linkage(X_reduced, method='single')

plt.figure(figsize=(10, 7))
sch.dendrogram(linkage_matrix)
plt.title('Dendrogram')
plt.xlabel('Samples')
plt.ylabel('Euclidean distances')
plt.show()

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score
import pandas as pd

results = []
K_range = range(2, 11) # Checking 2 to 10 clusters
X = X_reduced

for k in K_range:
    model = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = model.fit_predict(X)
    
    # Calculate scores
    sil = silhouette_score(X, labels)
    ch = calinski_harabasz_score(X, labels)
    
    results.append({'k': k, 'silhouette': sil, 'calinski': ch})

# Convert to DataFrame to find the winner
df_results = pd.DataFrame(results)
best_k = df_results.loc[df_results['silhouette'].idxmax(), 'k']
print(f"The optimal number of clusters based on Silhouette is: {best_k}")

df_results

In [ ]:
# we have 22 clusters, lets see
model = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels = model.fit_predict(X)

In [ ]:
l2patch = defaultdict(list)
for i in range(len(labels)):
    l2patch[labels[i]].append(patches[i])
l2pw = defaultdict(list)
for i in range(len(labels)):
    l2pw[labels[i]].append(scaled_pws[i].reshape(8,3,3))

In [ ]:
L = 0
for i in range(8):
    S([p for p in l2patch[L][i]], 15, ncols=8, viztype="global")
    plt.show()

In [ ]:
L = 1
for i in range(8):
    S([p for p in l2patch[L][i]], 15, ncols=8, viztype="local")
    plt.show()

# On the higher contrib ones

In [ ]:
d = SaliencyMap.objects.filter(coordinate="layers.2.out_12").first().data
out_h, out_w = len(d), len(d[0])
out_h, out_w

In [ ]:
# patches = []
patches = []
for alias, act in ip2act.items():
    sm = SaliencyMap.objects.filter(input__alias=alias, coordinate="layers.2.out_12").first()
    pos_vals = get_pos(sm)
    pos_vals = [p for p in pos_vals if p[0] > 0.02]
    grid_coords = [a[1] for a in pos_vals]
    grid_coords = [g for g in grid_coords if g in [(2,3), (2,4), (2,5)]]

    for (y,x) in grid_coords:
        (y0,x0), (y1,x1) = get_receptive(y,x)
        patches.append(act[:, y0:y1, x0:x1])
    

In [ ]:
from sklearn.preprocessing import StandardScaler

lin_patches = [p.reshape(-1) for p in patches]
scaler = StandardScaler()
pws = lin_patches * kernel.reshape(-1)
# scaled_pws = scaler.fit_transform(pws)
scaled_pws = pws

In [ ]:
from sklearn.decomposition import PCA

# 14 is fine, found after doing for 20
# weird that the value i need to set is higher actually
pca = PCA(11)
X_reduced = pca.fit_transform(lin_patches)


exp_var_pca = pca.explained_variance_ratio_
cum_sum_eigenvalues = np.cumsum(exp_var_pca)

# 5. Create the Scree Plot
plt.figure(figsize=(10, 6))

# Individual variance bars
plt.bar(range(1, len(exp_var_pca) + 1), exp_var_pca, alpha=0.5, align='center',
        label='Individual explained variance')

# Cumulative variance step plot
plt.step(range(1, len(cum_sum_eigenvalues) + 1), cum_sum_eigenvalues, where='mid',
         label='Cumulative explained variance', color='red')

plt.ylabel('Explained variance ratio')
plt.xlabel('Principal component index')
plt.title('Scree Plot: Explained Variance by Components')
plt.xticks(range(1, len(exp_var_pca) + 1))
plt.legend(loc='best')
plt.tight_layout()
plt.show()

In [ ]:
import scipy.cluster.hierarchy as sch
import matplotlib.pyplot as plt

# linkage performs the actual clustering
linkage_matrix = sch.linkage(X_reduced, method='single')

plt.figure(figsize=(10, 7))
sch.dendrogram(linkage_matrix)
plt.title('Dendrogram')
plt.xlabel('Samples')
plt.ylabel('Euclidean distances')
plt.show()

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score
import pandas as pd

results = []
K_range = range(2, 10) # Checking 2 to 10 clusters
X = X_reduced

for k in K_range:
    model = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = model.fit_predict(X)
    
    # Calculate scores
    sil = silhouette_score(X, labels)
    ch = calinski_harabasz_score(X, labels)
    
    results.append({'k': k, 'silhouette': sil, 'calinski': ch})

# Convert to DataFrame to find the winner
df_results = pd.DataFrame(results)
best_k = df_results.loc[df_results['silhouette'].idxmax(), 'k']
print(f"The optimal number of clusters based on Silhouette is: {best_k}")

df_results

In [ ]:
model = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels = model.fit_predict(X_reduced)

In [ ]:
l2patch = defaultdict(list)
for i in range(len(labels)):
    l2patch[labels[i]].append(patches[i])

l2pw = defaultdict(list)
for i in range(len(labels)):
    l2pw[labels[i]].append(scaled_pws[i].reshape(8,3,3))

In [ ]:
for l,v in l2patch.items():
    print(l, len(v))

In [ ]:
L = 0
for i in range(10):
    S([p for p in l2patch[L][i]], 15, ncols=8, viztype="global")
    plt.show()

In [ ]:
L = 1
for i in range(10):
    S([p for p in l2patch[L][i]], 15, ncols=8, viztype="global")
    plt.show()

In [ ]:
L = 2
for i in range(10):
    S([p for p in l2patch[L][i]], 15, ncols=8, viztype="global")
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

def plot_elbow_method(data, max_k=4):
    inertia = []
    k_values = range(1, max_k + 1)

    for k in k_values:
        # random_state ensures consistency across runs
        kmeans = KMeans(n_clusters=k, init='k-means++', max_iter=300, n_init=10, random_state=42)
        kmeans.fit(data)
        inertia.append(kmeans.inertia_)

    # 2. Plotting the results
    plt.figure(figsize=(10, 6))
    plt.plot(k_values, inertia, marker='o', linestyle='--', color='b')
    plt.title('The Elbow Method for Optimal K')
    plt.xlabel('Number of Clusters (k)')
    plt.ylabel('Inertia (Within-cluster Sum of Squares)')
    plt.xticks(k_values)
    plt.grid(True)
    plt.show()

# Run the function
plot_elbow_method(X_reduced, max_k=5)

In [ ]:
kmeans = KMeans(n_clusters=5, init='k-means++', max_iter=300, n_init=10, random_state=42)
kmeans.fit(X_reduced)

In [ ]:
len(labels)

In [ ]:
labels = kmeans.labels_
l2patch = defaultdict(list)
for i in range(len(labels)):
    l2patch[labels[i]].append(patches[i])

l2pw = defaultdict(list)
for i in range(len(labels)):
    l2pw[labels[i]].append(scaled_pws[i].reshape(8,3,3))

In [ ]:
for l,v in l2patch.items():
    print(l, len(v))

In [ ]:
L = 0
for i in range(10):
    S([p for p in l2patch[L][i]], 15, ncols=8, viztype="global")
    plt.show()

In [ ]:
from sklearn.cluster import HDBSCAN

dbscan = HDBSCAN(4)
dbscan.fit(X)

In [ ]:
dbscan.labels_

In [ ]:
labels = dbscan.labels_
l2patch = defaultdict(list)
for i in range(len(labels)):
    l2patch[labels[i]].append(patches[i])

l2pw = defaultdict(list)
for i in range(len(labels)):
    l2pw[labels[i]].append(scaled_pws[i].reshape(8,3,3))

## Lots of vis

In [ ]:
L = 0
for i in range(10):
    S([p for p in l2patch[L][i]], 15, ncols=8, viztype="global")
    plt.show()

In [ ]:
L = 1
for i in range(10):
    S([p for p in l2patch[L][i]], 15, ncols=8, viztype="global")
    plt.show()

In [ ]:
L = 2
for i in range(10):
    S([p for p in l2patch[L][i]], 15, ncols=8, viztype="global")
    plt.show()

In [ ]:
L = 3
for i in range(10):
    S([p for p in l2patch[L][i]], 15, ncols=8, viztype="global")
    plt.show()

In [ ]:
# patches = []
patches = []
for alias, act in ip2act.items():
    sm = SaliencyMap.objects.filter(input__alias=alias, coordinate="layers.2.out_12").first()
    pos_vals = get_pos(sm)
    pos_vals = [p for p in pos_vals if p[0] > 0.02]
    grid_coords = [a[1] for a in pos_vals]
    # grid_coords = [g for g in grid_coords if g in [(2,3), (2,4), (2,5)]]
    grid_coords = [g for g in grid_coords if g in [(2,4), ]]

    for (y,x) in grid_coords:
        (y0,x0), (y1,x1) = get_receptive(y,x)
        patches.append(act[:, y0:y1, x0:x1])

In [ ]:
patches[0].shape

In [ ]:
pws = [p * kernel for p in patches]

In [ ]:
mean_pws = sum(pws) / len(pws)

In [ ]:
std = np.std(np.stack(pws), axis=0)
S([pw for pw in std], 15, ncols=8, viztype="global")

In [ ]:
S([pw for pw in mean_pws], 15, ncols=8, viztype="global")

In [ ]:
S([pw for pw in mean_pws], 15, ncols=8, viztype="local")

In [ ]:
(patches[0] * kernel).shape

In [ ]:
from sklearn.cluster import HDBSCAN

dbscan = HDBSCAN(4)
lin_patches = [p.reshape(-1) for p in patches]
lin_pws = [p * kernel[6].reshape(-1) for p in lin_patches]
# scaled_pws = scaler.fit_transform(lin_pws)
dbscan.fit(lin_pws)

In [ ]:
np.unique(dbscan.labels_, return_counts=True)

In [ ]:
labels = dbscan.labels_
l2patch = defaultdict(list)
for i in range(len(labels)):
    l2patch[labels[i]].append(patches[i])

l2pw = defaultdict(list)
for i in range(len(labels)):
    l2pw[labels[i]].append(lin_pws[i].reshape(3,3))

In [ ]:
S(l2patch[0][:10], 15, ncols=8, viztype="global")
plt.show()

In [ ]:
S(l2patch[1][:10], 15, ncols=5, viztype="global")
plt.show()

In [ ]:
S(l2patch[3][:10], 15, ncols=9, viztype="global")
plt.show()

In [ ]:
S(l2patch[4][:10], 15, ncols=4, viztype="global")
plt.show()

In [ ]:
S(l2patch[5][:10], 15, ncols=8, viztype="global")
plt.show()

In [ ]:
S(l2patch[6][:10], 15, ncols=10, viztype="global")
plt.show()

In [ ]:
S(l2patch[7][:10], 15, ncols=10, viztype="global")
plt.show()

In [ ]:
len(l2patch[-1])

In [ ]:
S(l2pw[-1][:40], (15,6), ncols=10, viztype="global")
plt.show()

In [ ]:
S([kernel[6]])

# Higher contribs, filtered input

In [ ]:
alias = "4-Sl1"
act = ip2act[alias]
coord = (2,3)
print(gt_002[0][2].input.alias, gt_002[0][1])

In [ ]:
S([a for a in act], (15,5), 8)
plt.show()

In [ ]:
# first pick a poi with high val actually
print(coord)
(y0,x0), (y1,x1) = get_receptive(*coord)

patch = act[:, y0:y1, x0:x1]
S([a for a in patch], (15,5), 8)
plt.show()

In [ ]:
def make_sparse(patch, num_std=1):
    mean, std = np.mean(patch), np.std(patch)
    thresh = mean + num_std * std

    sparse = patch.copy()
    sparse[np.abs(sparse) < thresh] = 0
    return sparse

In [ ]:
# patches = []
patches = []
for alias, act in ip2act.items():
    sm = SaliencyMap.objects.filter(input__alias=alias, coordinate="layers.2.out_12").first()
    pos_vals = get_pos(sm)
    pos_vals = [p for p in pos_vals if p[0] > 0.02]
    grid_coords = [a[1] for a in pos_vals]
    grid_coords = [g for g in grid_coords if g in [(2,3), (2,4), (2,5)]]

    for (y,x) in grid_coords:
        (y0,x0), (y1,x1) = get_receptive(y,x)
        patches.append(act[:, y0:y1, x0:x1])
    

In [ ]:
lin_patches = [p.reshape(-1) for p in patches]
pws = lin_patches * kernel.reshape(-1)
sparse_pws = [make_sparse(pw) for pw in pws]

In [ ]:
S([p for p in pws[1].reshape(8,3,3)], 15, 8)
plt.show()
S([p for p in sparse_pws[1].reshape(8,3,3)], 15, 8)
plt.show()

In [ ]:
X = do_pca(12, sparse_pws)

In [ ]:
from sklearn.cluster import HDBSCAN

dbscan = HDBSCAN()
dbscan.fit(X)

In [ ]:
labels = dbscan.labels_
l2patch = defaultdict(list)

for i in range(len(labels)):
    l2patch[labels[i]].append(patches[i])

l2sparse = defaultdict(list)
for i in range(len(labels)):
    l2sparse[labels[i]].append(sparse_pws[i].reshape(8,3,3))

l2pw = defaultdict(list)
for i in range(len(labels)):
    l2pw[labels[i]].append(pws[i].reshape(8,3,3))

In [ ]:
for l,v in l2pw.items():
    print(l, len(v))

In [ ]:
L = 0
for i in range(5):
    S([p for p in l2sparse[L][i]], (15,5), ncols=8, viztype="global")
    plt.show()

In [ ]:
L = 1
for i in range(5):
    S([p for p in l2sparse[L][i]], (15,5), ncols=8, viztype="global")
    plt.show()

In [ ]:
L = -1
for i in range(5):
    S([p for p in l2sparse[L][i]], (15,5), ncols=8, viztype="global")
    plt.show()

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score
import pandas as pd

results = []
K_range = range(2, 10) # Checking 2 to 10 clusters

for k in K_range:
    model = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = model.fit_predict(X)
    
    # Calculate scores
    sil = silhouette_score(X, labels)
    ch = calinski_harabasz_score(X, labels)
    
    results.append({'k': k, 'silhouette': sil, 'calinski': ch})

# Convert to DataFrame to find the winner
df_results = pd.DataFrame(results)
best_k = df_results.loc[df_results['silhouette'].idxmax(), 'k']
print(f"The optimal number of clusters based on Silhouette is: {best_k}")

df_results

In [ ]:
labels = dbscan.labels_
l2patch = defaultdict(list)

for i in range(len(labels)):
    l2patch[labels[i]].append(patches[i])

l2sparse = defaultdict(list)
for i in range(len(labels)):
    l2sparse[labels[i]].append(sparse_pws[i].reshape(8,3,3))

l2pw = defaultdict(list)
for i in range(len(labels)):
    l2pw[labels[i]].append(pws[i].reshape(8,3,3))

In [ ]:
L = 0
for i in range(5):
    S([p for p in l2sparse[L][i]], (15,5), ncols=8, viztype="global")
    plt.show()

In [ ]:
L = 1
for i in range(5):
    S([p for p in l2sparse[L][i]], (15,5), ncols=8, viztype="global")
    plt.show()

In [ ]:
L = 2
for i in range(5):
    S([p for p in l2sparse[L][i]], (15,5), ncols=8, viztype="global")
    plt.show()